# 365 Probabilidades — Dia #008
## Qual a probabilidade de você transmitir mais características suas para seus filhos do que imagina?

**Tipo:** Preditivo  
**Data de publicação:** 2026-06-21  
**Ferramenta:** Python  
**Decisão analisada:** O quanto da minha personalidade e inteligência os meus filhos herdam?  
**Hashtag:** #365Probabilidades #Dia008

---

### 📖 A História

Todo pai e toda mãe já se perguntou isso. Quando o filho faz uma birra idêntica à sua, quando ri do mesmo jeito, quando tem o mesmo medo de altura, quando escolhe a mesma profissão sem que ninguém sugerisse.

A ciência passou décadas tentando separar o que vem dos genes e o que vem do ambiente. Estudando gêmeos idênticos criados em famílias diferentes. Acompanhando crianças adotadas que nunca conheceram os pais biológicos. Comparando irmãos criados juntos e separados.

O resultado é um dos achados mais replicados da psicologia: você transmite muito mais do que imagina. Não só a cor dos olhos e a altura. Personalidade. Inteligência. Ansiedade. Criatividade. Propensão à felicidade.

E há um fenômeno matemático que explica por que filhos de pais ou mães excepcionais raramente são tão excepcionais quanto os pais — mas ainda são melhores que a média. Francis Galton chamou isso de **regressão à média**. O modelo de hoje quantifica exatamente isso.

---

### 📚 O Conceito: Hereditariedade e Regressão à Média

**Hereditariedade** é a proporção da variação em um traço atribuída a diferenças genéticas. Uma hereditariedade de 66% na inteligência adulta significa que 2/3 da diferença entre as pessoas em inteligência é de origem genética.

**Regressão à média:** dado que um filho herda 50% dos genes de cada pai, a correlação esperada entre pai e filho para um traço é aproximadamente h²/2 — onde h² é a hereditariedade. Para inteligência (h²=0,66), essa correlação é 0,33. Isso significa que se um pai ou mãe está 1 desvio-padrão acima da média em inteligência, o filho estará em média 0,33dp acima — e terá 62,9% de chance de superar a média populacional.

---

### 🧮 O Modelo
**Fontes:**
- Polderman et al. (2015) — *Meta-analysis of the heritability of human traits* — Nature Genetics — N=14,5 milhões de pares de gêmeos, 39 países
- Plomin et al. (2016) — *Top 10 Replicated Findings From Behavioral Genetics* — Perspectives on Psychological Science — N=11.000 pares de gêmeos
- Vukasović & Bratko (2015) — *Heritability of personality: A meta-analysis* — Psychological Bulletin — N>100.000


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print('✅ Bibliotecas carregadas')

✅ Bibliotecas carregadas


In [2]:
# --- DADOS DA LITERATURA ---

# Polderman et al. 2015 — N=14,5M pares de gemeos, 39 paises
p_heredit_media   = 0.49
n_polderman       = 14500000

# Plomin et al. 2016 — N=11.000 pares de gemeos, 4 paises
h2_intelig_infancia = 0.41
h2_intelig_adulto   = 0.66
n_plomin            = 11000

# Vukasovic & Bratko 2015 — N>100.000
h2_personalidade    = 0.40
n_personalidade     = 100000

fator_correcao = 0.80

# Correlacao pai-filho = h2 / 2 (filho herda 50% dos genes do pai)
r_pai_filho_intel  = h2_intelig_adulto  / 2   # 0.33
r_pai_filho_person = h2_personalidade   / 2   # 0.20

print('=' * 65)
print('  DADOS DA LITERATURA — HEREDITARIEDADE')
print('=' * 65)
print(f'\n  Polderman et al. 2015 (N={n_polderman:,}):')
print(f'  → Hereditariedade media de tracos psicologicos: {p_heredit_media*100:.0f}%')
print(f'\n  Plomin et al. 2016 (N={n_plomin:,}):')
print(f'  → Inteligencia na infancia (9 anos)  : {h2_intelig_infancia*100:.0f}%')
print(f'  → Inteligencia na vida adulta         : {h2_intelig_adulto*100:.0f}%')
print(f'  → Correlacao pai-filho (inteligencia) : r = {r_pai_filho_intel:.2f}')
print(f'\n  Vukasovic & Bratko 2015 (N>{n_personalidade:,}):')
print(f'  → Hereditariedade da personalidade   : {h2_personalidade*100:.0f}%')
print(f'  → Correlacao pai-filho (personalidade): r = {r_pai_filho_person:.2f}')
print('=' * 65)

  DADOS DA LITERATURA — HEREDITARIEDADE

  Polderman et al. 2015 (N=14,500,000):
  → Hereditariedade media de tracos psicologicos: 49%

  Plomin et al. 2016 (N=11,000):
  → Inteligencia na infancia (9 anos)  : 41%
  → Inteligencia na vida adulta         : 66%
  → Correlacao pai-filho (inteligencia) : r = 0.33

  Vukasovic & Bratko 2015 (N>100,000):
  → Hereditariedade da personalidade   : 40%
  → Correlacao pai-filho (personalidade): r = 0.20


In [6]:
# --- O MODELO ---
# Parte 1: Distribuicao Beta para hereditariedade media
alpha_m = p_heredit_media * n_polderman
beta_m  = (1 - p_heredit_media) * n_polderman
dist_m  = stats.beta(alpha_m, beta_m)
ic_m    = dist_m.interval(0.95)

# Parte 2: Probabilidade de transmissao de traco especifico
# Dado pai ou mãe a X desvios-padrao da media, qual P(filho > media)?
# E(filho | pai = X dp) = r_pai_filho * X dp
# P(filho > media) = P(Z > -r*X) = 1 - Phi(-r*X)

def p_filho_acima_media(r, z_pai):
    e_filho = r * z_pai
    return 1 - stats.norm.cdf(-e_filho)

# Cenarios para inteligencia
z_pais = [0, 0.5, 1.0, 1.5, 2.0]  # desvios-padrao acima da media
p_intel = [p_filho_acima_media(r_pai_filho_intel, z) for z in z_pais]
p_person = [p_filho_acima_media(r_pai_filho_person, z) for z in z_pais]

p_media_corrigido = p_heredit_media * fator_correcao

print('=' * 65)
print('  MODELO — PROBABILIDADE DE TRANSMISSAO')
print('=' * 65)
print(f'\n  Hereditariedade media — IC 95%:')
print(f'  → [{ic_m[0]*100:.2f}%, {ic_m[1]*100:.2f}%]')
print(f'  → Corrigido (x0.80): {p_media_corrigido*100:.1f}%')
print()
print(f'  Probabilidade do filho superar a media populacional:')
print(f'  {"Pai/Mae (dp acima media)":<25} {"Inteligencia":<18} {"Personalidade"}')
print(f'  {"-"*60}')
for z, pi, pp in zip(z_pais, p_intel, p_person):
    pct_pai = (1 - stats.norm.cdf(-z)) * 100
    print(f'  +{z:.1f}dp (top {100-pct_pai:.0f}%)         {pi*100:.1f}%             {pp*100:.1f}%')
print()
print(f'  Ganho vs baseline (pai na media, P=50%):')
print(f'  → Inteligencia (pai/mãe +1dp): +{(p_intel[2]-0.50)*100:.1f}pp')
print(f'  → Personalidade (pai/mãe +1dp): +{(p_person[2]-0.50)*100:.1f}pp')
print('=' * 65)

  MODELO — PROBABILIDADE DE TRANSMISSAO

  Hereditariedade media — IC 95%:
  → [48.97%, 49.03%]
  → Corrigido (x0.80): 39.2%

  Probabilidade do filho superar a media populacional:
  Pai/Mae (dp acima media)  Inteligencia       Personalidade
  ------------------------------------------------------------
  +0.0dp (top 50%)         50.0%             50.0%
  +0.5dp (top 31%)         56.6%             54.0%
  +1.0dp (top 16%)         62.9%             57.9%
  +1.5dp (top 7%)         69.0%             61.8%
  +2.0dp (top 2%)         74.5%             65.5%

  Ganho vs baseline (pai na media, P=50%):
  → Inteligencia (pai/mãe +1dp): +12.9pp
  → Personalidade (pai/mãe +1dp): +7.9pp


In [4]:
# --- VISUALIZACAO ---

# GRAFICO 1 — Probabilidade de transmissao por cenario de pai
fig1, ax1 = plt.subplots(figsize=(12, 8))

z_range = np.linspace(0, 2.5, 100)
p_intel_range  = [p_filho_acima_media(r_pai_filho_intel, z) for z in z_range]
p_person_range = [p_filho_acima_media(r_pai_filho_person, z) for z in z_range]

ax1.plot(z_range, [p * 100 for p in p_intel_range],
         color='#c8a84b', linewidth=3, label=f'Inteligencia (h²={h2_intelig_adulto}, r={r_pai_filho_intel:.2f})')
ax1.plot(z_range, [p * 100 for p in p_person_range],
         color='#2a8a82', linewidth=3, linestyle='--', label=f'Personalidade (h²={h2_personalidade}, r={r_pai_filho_person:.2f})')
ax1.axhline(y=50, color='#888888', linestyle=':', linewidth=1.5, label='Baseline: 50% (pais na media)')

# Marcadores nos pontos chave
for z, pi, pp in zip([1.0, 2.0], [p_intel[2], p_intel[4]], [p_person[2], p_person[4]]):
    ax1.scatter([z], [pi * 100], color='#c8a84b', s=80, zorder=5)
    ax1.scatter([z], [pp * 100], color='#2a8a82', s=80, zorder=5)
    ax1.annotate(f'{pi*100:.1f}%', xy=(z, pi*100), xytext=(8, 4),
                 textcoords='offset points', color='#c8a84b', fontweight='bold', fontsize=10)
    ax1.annotate(f'{pp*100:.1f}%', xy=(z, pp*100), xytext=(8, -14),
                 textcoords='offset points', color='#2a8a82', fontweight='bold', fontsize=10)

ax1.set_xlabel('Posicao do pai/mae em relacao a media populacional (desvios-padrao)', fontsize=12)
ax1.set_ylabel('P(filho superar a media) %', fontsize=12)
ax1.set_title('Probabilidade de transmissao — Regressao a Media de Galton\nPlomin et al. 2016 · Vukasovic & Bratko 2015',
              fontsize=13, pad=15)
ax1.legend(fontsize=11)
ax1.set_xlim(0, 2.5)
ax1.set_ylim(45, 80)

plt.tight_layout()
plt.savefig('dia-008-grafico-01-transmissao.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Grafico 1 salvo!')


# GRAFICO 2 — Hereditariedade por traco
fig2, ax2 = plt.subplots(figsize=(12, 8))

tracos  = ['Altura', 'Inteligencia\n(adulto)', 'Media tracos\npsicologicos',
           'Inteligencia\n(infancia)', 'Personalidade', 'Felicidade\nsubjetiva']
heredit = [0.80, 0.66, 0.49, 0.41, 0.40, 0.36]
cores2  = ['#888888', '#c8a84b', '#c0392b', '#c8a84b', '#2a8a82', '#2a8a82']

bars = ax2.barh(tracos, [h * 100 for h in heredit], color=cores2, alpha=0.85, height=0.55)
ax2.axvline(x=50, color='#c0392b', linestyle='--', linewidth=1.5, label='50% — metade da variacao e genetica')
ax2.set_xlabel('Hereditariedade (%)', fontsize=12)
ax2.set_xlim(0, 100)
ax2.set_title('O quanto cada traco e herdado geneticamente\nPolderman et al. 2015 · Plomin et al. 2016', fontsize=13, pad=15)
ax2.legend(fontsize=10)

for bar, h in zip(bars, heredit):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
             f'{h*100:.0f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('dia-008-grafico-02-tracos.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Grafico 2 salvo!')


# GRAFICO 3 — Distribuicao Beta (hereditariedade media)
fig3, ax3 = plt.subplots(figsize=(12, 8))

x = np.linspace(ic_m[0] * 0.9999, ic_m[1] * 1.0001, 1000)
y = dist_m.pdf(x)

ax3.plot(x * 100, y, color='#c0392b', linewidth=3)
ax3.fill_between(x * 100, y, alpha=0.2, color='#c0392b')
ax3.axvline(x=ic_m[0] * 100, color='#c8a84b', linestyle='--', linewidth=2,
            label=f'IC 95%: [{ic_m[0]*100:.2f}%, {ic_m[1]*100:.2f}%]')
ax3.axvline(x=ic_m[1] * 100, color='#c8a84b', linestyle='--', linewidth=2)
ax3.axvline(x=p_heredit_media * 100, color='#c0392b', linewidth=2,
            label=f'Estimativa central: {p_heredit_media*100:.0f}%')
ax3.axvline(x=p_media_corrigido * 100, color='#2a8a82', linestyle='-.',
            linewidth=2, label=f'Corrigido (x0.80): {p_media_corrigido*100:.1f}%')

ax3.set_xlabel('Hereditariedade media de tracos psicologicos (%)', fontsize=12)
ax3.set_ylabel('Densidade', fontsize=12)
ax3.set_title('Distribuicao Beta — Hereditariedade media\nPolderman et al. (2015) — N=14,5 milhoes de pares de gemeos', fontsize=13, pad=15)
ax3.legend(fontsize=11)

plt.tight_layout()
plt.savefig('dia-008-grafico-03-bayesiano.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Grafico 3 salvo!')

✅ Grafico 1 salvo!
✅ Grafico 2 salvo!
✅ Grafico 3 salvo!


### 💡 O Insight

Em meta-análise com **14,5 milhões de pares de gêmeos em 39 países**, cerca de **49% da variação** em traços psicológicos é explicada por genes. Para inteligência na vida adulta, esse número sobe para **66%**.

O modelo vai além: se um pai ou mãe está 1 desvio-padrão acima da média em inteligência (top 16%), o filho tem **62,9%** de chance de superar a média populacional — contra 50% de qualquer filho de pai mediano. Um ganho de +12,9 pontos percentuais apenas por herança genética.

É a regressão à média de Galton: filhos de pais ou mães excepcionais raramente são tão excepcionais quanto os pais — mas carregam uma vantagem real e mensurável. E o reverso também é verdade: filhos de pais ou mães com dificuldades tendem a superar os pais, mas ainda partem de uma posição menos favorável que a média.

*Quais características suas você quer que seus filhos herdem — e quais você preferiria que não fossem transmitidas?*

---

### ⚠️ Limitações do Modelo
- Hereditariedade é uma estimativa populacional — não determina o destino de um indivíduo específico
- O modelo de regressão à média assume herança aditiva simples — a realidade genética é mais complexa (interações, epigenética)
- Os valores variam por traço, população e contexto cultural
- Os estudos mais robustos foram conduzidos principalmente em países de alta renda
- O fator de correção (×0,80) é uma aproximação padrão do projeto

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*